# Market Data

Download the observed AAPL trades used by the preprocessing workflow, then inspect the normalized project schema. Existing Parquet data is reused so repeated notebook runs do not make unnecessary external requests.

## Download the Data

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.data_preprocessing.market_data import save_alpaca_historical_data

PROJECT_ROOT = Path.cwd().resolve().parents[1]
market_path = PROJECT_ROOT / "data/research_data/market/data/aapl_2025-01-01_2025-12-31.parquet"

if not market_path.is_file():
    save_alpaca_historical_data(
        symbols=["AAPL"],
        start=datetime(2025, 1, 1, tzinfo=timezone.utc),
        end=datetime(2025, 12, 31, 23, 59, 59, 999999, tzinfo=timezone.utc),
        asset_class="stock",
        data_type="tick",
        output_path=market_path,
    )

market_data = pd.read_parquet(market_path)
expected_columns = ["timestamp", "symbol", "price", "size"]
assert market_data.columns.tolist() == expected_columns
market_data["timestamp"] = pd.to_datetime(market_data["timestamp"], utc=True)
assert market_data["symbol"].eq("AAPL").all()
market_path

## Take a Quick Look at the Data Structure

In [ ]:
market_data.head()

In [ ]:
market_data.info()

In [ ]:
market_data["symbol"].value_counts(dropna=False)

In [ ]:
market_data[["price", "size"]].describe()

In [ ]:
market_data[["price", "size"]].hist(bins=50, figsize=(10, 4))
plt.tight_layout()
plt.show()